# YOLO26 Vessel Detection Experiments with SAHI

This notebook systematically tests different configurations to find the best model settings for Jupiter Inlet vessel detection.

**Experimental Design:** Change only ONE variable at a time to isolate its effect.

**Variables tested:**
- Image size: 512, 640, 768, 1024
- Epochs: 50, 100, 150, 200
- Model size: nano, small, medium, large
- SAHI inference: Compare standard vs SAHI inference for each trained model

## 1. Setup and Configuration

In [1]:
# Install/upgrade packages if needed
!pip install -U ultralytics sahi

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from ultralytics import YOLO
import torch
import time
from datetime import datetime
from pathlib import Path
import os

# Verify GPU is available
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA B200


In [3]:
# Check SAHI installation
try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction, get_prediction
    from sahi.utils.coco import Coco
    print("✅ SAHI installed and ready")
except ImportError:
    print("❌ SAHI not installed. Run: pip install sahi")

✅ SAHI installed and ready


In [4]:
# =============================================================================
# CONFIGURE YOUR PATHS HERE
# =============================================================================

data_yaml_path = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml"

# For SAHI evaluation - path to validation images and annotations
val_images_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/val"
val_coco_json = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/_annotations.coco.json"  # Update if different

# Verify paths exist
for path, name in [(data_yaml_path, "Data config"), (val_images_dir, "Val images")]:
    if Path(path).exists():
        print(f"✅ {name} found: {path}")
    else:
        print(f"❌ {name} NOT found: {path}")

✅ Data config found: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml
✅ Val images found: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/val


## 2. Define Base Configuration

This is the "control" configuration. Each experiment will change only ONE parameter from this baseline.

In [5]:
# Base configuration - the "control" for all experiments
BASE_CONFIG = {
    'data': data_yaml_path,
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'rect': True,
    'device': 0,  # Use GPU
    
    # Augmentations
    'mosaic': 0.5,
    'scale': 0.2,
    'fliplr': 0.5,
    'flipud': 0.0,
    
    # Training settings
    'patience': 50,
    'save': True,
    'plots': True,
    'project': 'jupiter_inlet_experiments',
}

BASE_MODEL = 'yolo26s.pt'

# SAHI settings
SAHI_OVERLAP_RATIO = 0.2  # 20% overlap between slices
SAHI_CONFIDENCE_THRESHOLD = 0.25

print("Base configuration:")
for k, v in BASE_CONFIG.items():
    print(f"  {k}: {v}")
print(f"  model: {BASE_MODEL}")
print(f"\nSAHI settings:")
print(f"  overlap_ratio: {SAHI_OVERLAP_RATIO}")
print(f"  confidence_threshold: {SAHI_CONFIDENCE_THRESHOLD}")

Base configuration:
  data: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  rect: True
  device: 0
  mosaic: 0.5
  scale: 0.2
  fliplr: 0.5
  flipud: 0.0
  patience: 50
  save: True
  plots: True
  project: jupiter_inlet_experiments
  model: yolo26s.pt

SAHI settings:
  overlap_ratio: 0.2
  confidence_threshold: 0.25


## 3. Define Experiments

Each experiment changes **only one variable** from the baseline.

For each trained model, we'll evaluate with:
1. Standard inference
2. SAHI inference (slice size = training image size)

In [6]:
# Define all experiments
# Format: {'name': experiment_name, 'model': model_file, 'changes': {config changes}}

experiments = [
    # ==========================================================================
    # BASELINE
    # ==========================================================================
    {'name': 'baseline_s_640_e100', 'model': 'yolo26s.pt', 'changes': {}},
    
    # ==========================================================================
    # IMAGE SIZE EXPERIMENTS (keeping epochs=100, model=small)
    # ==========================================================================
    {'name': 'imgsz_512', 'model': 'yolo26s.pt', 'changes': {'imgsz': 512}},
    {'name': 'imgsz_768', 'model': 'yolo26s.pt', 'changes': {'imgsz': 768}},
    {'name': 'imgsz_1024', 'model': 'yolo26s.pt', 'changes': {'imgsz': 1024, 'batch': 8}},
    
    # ==========================================================================
    # EPOCH EXPERIMENTS (keeping imgsz=640, model=small)
    # ==========================================================================
    {'name': 'epochs_50', 'model': 'yolo26s.pt', 'changes': {'epochs': 50}},
    {'name': 'epochs_150', 'model': 'yolo26s.pt', 'changes': {'epochs': 150}},
    {'name': 'epochs_200', 'model': 'yolo26s.pt', 'changes': {'epochs': 200}},
    
    # ==========================================================================
    # MODEL SIZE EXPERIMENTS (keeping imgsz=640, epochs=100)
    # ==========================================================================
    {'name': 'model_nano', 'model': 'yolo26n.pt', 'changes': {}},
    {'name': 'model_medium', 'model': 'yolo26m.pt', 'changes': {}},
    {'name': 'model_large', 'model': 'yolo26l.pt', 'changes': {'batch': 8}},
]

print(f"Total experiments to run: {len(experiments)}")
print(f"Each experiment will be evaluated with standard AND SAHI inference")
print(f"Total evaluations: {len(experiments) * 2}")
print("\nExperiment list:")
for i, exp in enumerate(experiments, 1):
    changes_str = str(exp['changes']) if exp['changes'] else 'baseline'
    imgsz = exp['changes'].get('imgsz', BASE_CONFIG['imgsz'])
    print(f"  {i}. {exp['name']}: {exp['model']} | {changes_str} | SAHI slice: {imgsz}")

Total experiments to run: 10
Each experiment will be evaluated with standard AND SAHI inference
Total evaluations: 20

Experiment list:
  1. baseline_s_640_e100: yolo26s.pt | baseline | SAHI slice: 640
  2. imgsz_512: yolo26s.pt | {'imgsz': 512} | SAHI slice: 512
  3. imgsz_768: yolo26s.pt | {'imgsz': 768} | SAHI slice: 768
  4. imgsz_1024: yolo26s.pt | {'imgsz': 1024, 'batch': 8} | SAHI slice: 1024
  5. epochs_50: yolo26s.pt | {'epochs': 50} | SAHI slice: 640
  6. epochs_150: yolo26s.pt | {'epochs': 150} | SAHI slice: 640
  7. epochs_200: yolo26s.pt | {'epochs': 200} | SAHI slice: 640
  8. model_nano: yolo26n.pt | baseline | SAHI slice: 640
  9. model_medium: yolo26m.pt | baseline | SAHI slice: 640
  10. model_large: yolo26l.pt | {'batch': 8} | SAHI slice: 640


## 4. Helper Functions for SAHI Evaluation

In [7]:
def evaluate_with_sahi(model_path, val_images_dir, slice_size, overlap_ratio=0.2, conf_thresh=0.25):
    """
    Evaluate a trained model using SAHI sliced inference.
    
    Args:
        model_path: Path to trained YOLO model (best.pt)
        val_images_dir: Directory containing validation images
        slice_size: Size of slices (should match training imgsz)
        overlap_ratio: Overlap between slices (0.0-1.0)
        conf_thresh: Confidence threshold for detections
    
    Returns:
        dict with detection counts and timing info
    """
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    
    # Load model for SAHI
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=str(model_path),
        confidence_threshold=conf_thresh,
        device='cuda:0' if torch.cuda.is_available() else 'cpu'
    )
    
    # Get all validation images
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = [f for f in Path(val_images_dir).iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    total_detections = 0
    start_time = time.time()
    
    for img_path in image_files:
        result = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=slice_size,
            slice_width=slice_size,
            overlap_height_ratio=overlap_ratio,
            overlap_width_ratio=overlap_ratio,
        )
        total_detections += len(result.object_prediction_list)
    
    elapsed_time = time.time() - start_time
    
    return {
        'total_detections': total_detections,
        'num_images': len(image_files),
        'avg_detections_per_image': total_detections / len(image_files) if image_files else 0,
        'inference_time': elapsed_time,
        'time_per_image': elapsed_time / len(image_files) if image_files else 0
    }


def evaluate_standard(model_path, val_images_dir, imgsz, conf_thresh=0.25):
    """
    Evaluate a trained model using standard (non-SAHI) inference.
    
    Returns:
        dict with detection counts and timing info
    """
    model = YOLO(str(model_path))
    
    # Get all validation images
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = [f for f in Path(val_images_dir).iterdir() 
                   if f.suffix.lower() in image_extensions]
    
    total_detections = 0
    start_time = time.time()
    
    for img_path in image_files:
        results = model.predict(
            str(img_path),
            imgsz=imgsz,
            conf=conf_thresh,
            verbose=False
        )
        total_detections += len(results[0].boxes)
    
    elapsed_time = time.time() - start_time
    
    return {
        'total_detections': total_detections,
        'num_images': len(image_files),
        'avg_detections_per_image': total_detections / len(image_files) if image_files else 0,
        'inference_time': elapsed_time,
        'time_per_image': elapsed_time / len(image_files) if image_files else 0
    }

## 5. Quick Test (Optional)

Run a quick test with 3 epochs to make sure everything works before the full experiment.

In [9]:
# OPTIONAL: Quick test run (set to True to run)
RUN_QUICK_TEST = True

if RUN_QUICK_TEST:
    print("Running quick test with 3 epochs...")
    test_model = YOLO('yolo26s.pt')
    test_results = test_model.train(
        data=data_yaml_path,
        epochs=3,
        imgsz=640,
        batch=16,
        device=0,
        project='jupiter_inlet_experiments',
        name='quick_test',
    )
    
    # Test SAHI on the quick model
    best_model_path = Path(test_results.save_dir) / 'weights' / 'best.pt'
    print(f"\nTesting SAHI inference on: {best_model_path}")
    
    sahi_results = evaluate_with_sahi(
        best_model_path, 
        val_images_dir, 
        slice_size=640,
        overlap_ratio=SAHI_OVERLAP_RATIO
    )
    print(f"SAHI detections: {sahi_results['total_detections']}")
    print("✅ Quick test completed successfully!")
else:
    print("Skipping quick test. Set RUN_QUICK_TEST = True to run.")

Running quick test with 3 epochs...
Ultralytics 8.4.14 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA B200, 182642MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=quick_test, nbs=

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 178.1±93.6 MB/s, size: 278.8 KB)
val: Scanning /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/val... 74 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 1.1Kit/s 0.1s
val: New cache created: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/val.cache


/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/quick_test/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/quick_test
Starting training for 3 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/3      4.52G      2.954      4.949   0.002164         14        640: 100% ━━━━━━━━━━━━ 19/19 1.2it/s 15.2s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.1s/it 3.3s11.4s
                   all         74        255    

## 6. Run All Experiments

⚠️ **This will take several hours!** Each experiment with 100 epochs takes ~10-20 minutes on GPU, plus SAHI evaluation.

In [8]:
# Select which experiments to run
# Set to None to run ALL experiments, or specify indices like [0, 1, 2]

EXPERIMENTS_TO_RUN = None  # Run all
# EXPERIMENTS_TO_RUN = [0]  # Run only baseline
# EXPERIMENTS_TO_RUN = [0, 1, 2, 3]  # Run baseline + image size experiments
# EXPERIMENTS_TO_RUN = [4, 5, 6]  # Run only epoch experiments
# EXPERIMENTS_TO_RUN = [7, 8, 9]  # Run only model size experiments

# Whether to run SAHI evaluation after each training
RUN_SAHI_EVALUATION = True

if EXPERIMENTS_TO_RUN is None:
    experiments_to_run = experiments
else:
    experiments_to_run = [experiments[i] for i in EXPERIMENTS_TO_RUN]

print(f"Will run {len(experiments_to_run)} experiment(s):")
for exp in experiments_to_run:
    print(f"  - {exp['name']}")
print(f"\nSAHI evaluation: {'Enabled' if RUN_SAHI_EVALUATION else 'Disabled'}")

Will run 10 experiment(s):
  - baseline_s_640_e100
  - imgsz_512
  - imgsz_768
  - imgsz_1024
  - epochs_50
  - epochs_150
  - epochs_200
  - model_nano
  - model_medium
  - model_large

SAHI evaluation: Enabled


In [ ]:
# Storage for results
results_summary = []

# Track total time
total_start_time = time.time()

print(f"Starting experiments at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

for i, exp in enumerate(experiments_to_run, 1):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT {i}/{len(experiments_to_run)}: {exp['name']}")
    print(f"Model: {exp['model']}")
    print(f"Changes from baseline: {exp['changes'] if exp['changes'] else 'None (baseline)'}")
    print(f"{'='*70}\n")
    
    # Build config: start with base, apply changes
    config = BASE_CONFIG.copy()
    config.update(exp['changes'])
    config['name'] = exp['name']
    
    # Get the image size for this experiment (for SAHI slice size)
    imgsz = config['imgsz']
    
    # Track time for this experiment
    exp_start_time = time.time()
    
    try:
        # =====================================================================
        # TRAINING
        # =====================================================================
        print(f"Training with imgsz={imgsz}...")
        model = YOLO(exp['model'])
        results = model.train(**config)
        
        train_time = time.time() - exp_start_time
        
        # Extract training metrics
        metrics = results.results_dict
        best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
        
        # Base result entry
        result_entry = {
            'experiment': exp['name'],
            'model': exp['model'],
            'imgsz': imgsz,
            'epochs': config['epochs'],
            'batch': config['batch'],
            'mAP50': metrics.get('metrics/mAP50(B)', None),
            'mAP50-95': metrics.get('metrics/mAP50-95(B)', None),
            'precision': metrics.get('metrics/precision(B)', None),
            'recall': metrics.get('metrics/recall(B)', None),
            'training_time': f"{train_time/60:.1f} min",
            'best_model_path': str(best_model_path),
            'status': 'success'
        }
        
        print(f"\n✅ Training completed in {train_time/60:.1f} min")
        print(f"   mAP50: {result_entry['mAP50']:.4f}")
        print(f"   Recall: {result_entry['recall']:.4f}")
        
        # =====================================================================
        # SAHI EVALUATION
        # =====================================================================
        if RUN_SAHI_EVALUATION:
            print(f"\n   Running SAHI evaluation (slice_size={imgsz})...")
            
            # Standard inference
            std_results = evaluate_standard(
                best_model_path,
                val_images_dir,
                imgsz=imgsz,
                conf_thresh=SAHI_CONFIDENCE_THRESHOLD
            )
            
            # SAHI inference
            sahi_results = evaluate_with_sahi(
                best_model_path,
                val_images_dir,
                slice_size=imgsz,
                overlap_ratio=SAHI_OVERLAP_RATIO,
                conf_thresh=SAHI_CONFIDENCE_THRESHOLD
            )
            
            # Add SAHI metrics to result
            result_entry['std_detections'] = std_results['total_detections']
            result_entry['std_time_per_img'] = f"{std_results['time_per_image']:.3f}s"
            result_entry['sahi_detections'] = sahi_results['total_detections']
            result_entry['sahi_time_per_img'] = f"{sahi_results['time_per_image']:.3f}s"
            result_entry['sahi_slice_size'] = imgsz
            result_entry['detection_increase'] = sahi_results['total_detections'] - std_results['total_detections']
            
            pct_increase = ((sahi_results['total_detections'] / std_results['total_detections']) - 1) * 100 if std_results['total_detections'] > 0 else 0
            result_entry['detection_increase_pct'] = f"{pct_increase:.1f}%"
            
            print(f"   Standard detections: {std_results['total_detections']}")
            print(f"   SAHI detections: {sahi_results['total_detections']} ({pct_increase:+.1f}%)")
        
    except Exception as e:
        exp_time = time.time() - exp_start_time
        print(f"\n❌ FAILED: {exp['name']} after {exp_time/60:.1f} min")
        print(f"   Error: {str(e)}")
        
        result_entry = {
            'experiment': exp['name'],
            'model': exp['model'],
            'imgsz': imgsz,
            'epochs': config['epochs'],
            'batch': config['batch'],
            'mAP50': None,
            'mAP50-95': None,
            'precision': None,
            'recall': None,
            'training_time': f"{exp_time/60:.1f} min",
            'status': f'failed: {str(e)[:50]}'
        }
    
    results_summary.append(result_entry)

# Total time
total_time = time.time() - total_start_time
print(f"\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETED")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")

Starting experiments at 2026-02-10 12:15:23

EXPERIMENT 1/10: baseline_s_640_e100
Model: yolo26s.pt
Changes from baseline: None (baseline)

Training with imgsz=640...
Ultralytics 8.4.14 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA B200, 182642MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, m

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 168.4±86.0 MB/s, size: 278.8 KB)
val: Scanning /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/val.cache... 74 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 4.0Mit/s 0.0s


/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/baseline_s_640_e1002/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/baseline_s_640_e1002
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.05G       3.05      4.466   0.004346          6        640: 100% ━━━━━━━━━━━━ 19/19 1.2it/s 16.3s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.3s/it 4.0s13.6s
                   all    

...

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 662.6±240.3 MB/s, size: 279.3 KB)
val: Scanning /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/val.cache... 74 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 74/74 3.6Mit/s 0.0s


/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/imgsz_1024/labels.jpg... 
Image sizes 1024 train, 1024 val
Using 8 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments/imgsz_1024
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.86G      2.584      5.176   0.002758          7       1024: 100% ━━━━━━━━━━━━ 37/37 2.1it/s 18.0s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.6s0.1ss
                   all         74        25

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      3.12G      1.664     0.5655   0.001441          4       1024: 100% ━━━━━━━━━━━━ 37/37 11.0it/s 3.3s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 23.6it/s 0.2s.3s
                   all         74        255      0.759       0.78       0.78      0.355

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/100      3.12G       1.66     0.5484   0.001416          9       1024: 100% ━━━━━━━━━━━━ 37/37 16.2it/s 2.3s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 24.5it/s 0.2s.3s
                   all         74        255      0.754      0.773      0.784      0.362

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     93/100      3.12G      1.691     0.5548   0.001444          8       1024: 100% ━

## 7. Results Summary

In [ ]:
# Create DataFrame for easy analysis
df = pd.DataFrame(results_summary)

# Display training results
print("\n" + "="*80)
print("TRAINING RESULTS SUMMARY")
print("="*80 + "\n")

# Format for display
train_cols = ['experiment', 'model', 'imgsz', 'epochs', 'mAP50', 'recall', 'precision', 'training_time']
train_cols = [c for c in train_cols if c in df.columns]
df_train = df[train_cols].copy()

# Round numeric columns
for col in ['mAP50', 'recall', 'precision']:
    if col in df_train.columns:
        df_train[col] = df_train[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")

print(df_train.to_string(index=False))

In [ ]:
# Display SAHI comparison results
if RUN_SAHI_EVALUATION and 'sahi_detections' in df.columns:
    print("\n" + "="*80)
    print("SAHI vs STANDARD INFERENCE COMPARISON")
    print("="*80 + "\n")
    
    sahi_cols = ['experiment', 'imgsz', 'sahi_slice_size', 'std_detections', 'sahi_detections', 
                 'detection_increase', 'detection_increase_pct', 'std_time_per_img', 'sahi_time_per_img']
    sahi_cols = [c for c in sahi_cols if c in df.columns]
    df_sahi = df[sahi_cols].copy()
    
    print(df_sahi.to_string(index=False))
    
    # Summary stats
    print("\n" + "-"*40)
    avg_increase = df['detection_increase'].mean()
    print(f"Average detection increase with SAHI: {avg_increase:.1f} detections")

In [ ]:
# Find best performing experiments
df_success = df[df['status'] == 'success'].copy()

if len(df_success) > 0:
    print("\n" + "="*60)
    print("BEST PERFORMERS")
    print("="*60)
    
    # Best by mAP50
    best_map = df_success.loc[df_success['mAP50'].idxmax()]
    print(f"\n🏆 Best mAP50: {best_map['experiment']}")
    print(f"   mAP50: {best_map['mAP50']:.4f}")
    print(f"   Config: {best_map['model']}, imgsz={best_map['imgsz']}, epochs={best_map['epochs']}")
    
    # Best by Recall
    best_recall = df_success.loc[df_success['recall'].idxmax()]
    print(f"\n🏆 Best Recall: {best_recall['experiment']}")
    print(f"   Recall: {best_recall['recall']:.4f}")
    print(f"   Config: {best_recall['model']}, imgsz={best_recall['imgsz']}, epochs={best_recall['epochs']}")
    
    # Best SAHI improvement
    if 'detection_increase' in df_success.columns:
        best_sahi = df_success.loc[df_success['detection_increase'].idxmax()]
        print(f"\n🏆 Best SAHI Improvement: {best_sahi['experiment']}")
        print(f"   Extra detections: +{best_sahi['detection_increase']} ({best_sahi['detection_increase_pct']})")
        print(f"   Slice size: {best_sahi['sahi_slice_size']}")
else:
    print("No successful experiments to analyze.")

In [ ]:
# Save results to CSV
output_dir = Path(BASE_CONFIG['project'])
output_dir.mkdir(exist_ok=True)

output_path = output_dir / 'experiment_results.csv'
df.to_csv(output_path, index=False)
print(f"\n📁 Results saved to: {output_path}")

# Also save to a timestamped file as backup
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_path = output_dir / f'experiment_results_{timestamp}.csv'
df.to_csv(backup_path, index=False)
print(f"📁 Backup saved to: {backup_path}")

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Only plot successful experiments
df_plot = df[df['status'] == 'success'].copy()

if len(df_plot) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # mAP50 comparison
    axes[0].barh(df_plot['experiment'], df_plot['mAP50'], color='steelblue')
    axes[0].set_xlabel('mAP50')
    axes[0].set_title('mAP50 by Experiment')
    axes[0].axvline(x=df_plot['mAP50'].mean(), color='red', linestyle='--', label='Mean')
    
    # Recall comparison
    axes[1].barh(df_plot['experiment'], df_plot['recall'], color='forestgreen')
    axes[1].set_xlabel('Recall')
    axes[1].set_title('Recall by Experiment')
    axes[1].axvline(x=df_plot['recall'].mean(), color='red', linestyle='--', label='Mean')
    
    # Precision comparison
    axes[2].barh(df_plot['experiment'], df_plot['precision'], color='darkorange')
    axes[2].set_xlabel('Precision')
    axes[2].set_title('Precision by Experiment')
    axes[2].axvline(x=df_plot['precision'].mean(), color='red', linestyle='--', label='Mean')
    
    plt.tight_layout()
    
    # Save figure
    fig_path = output_dir / 'experiment_comparison.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"📊 Figure saved to: {fig_path}")
    
    plt.show()
else:
    print("No successful experiments to plot.")

In [ ]:
# SAHI comparison plot
if RUN_SAHI_EVALUATION and 'sahi_detections' in df.columns and len(df_plot) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Detection counts: Standard vs SAHI
    x = np.arange(len(df_plot))
    width = 0.35
    
    axes[0].bar(x - width/2, df_plot['std_detections'], width, label='Standard', color='steelblue')
    axes[0].bar(x + width/2, df_plot['sahi_detections'], width, label='SAHI', color='coral')
    axes[0].set_xlabel('Experiment')
    axes[0].set_ylabel('Total Detections')
    axes[0].set_title('Standard vs SAHI Detections')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(df_plot['experiment'], rotation=45, ha='right')
    axes[0].legend()
    
    # Detection increase by slice size
    axes[1].bar(df_plot['experiment'], df_plot['detection_increase'], color='forestgreen')
    axes[1].set_xlabel('Experiment')
    axes[1].set_ylabel('Additional Detections (SAHI - Standard)')
    axes[1].set_title('SAHI Detection Improvement')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    # Add slice size labels
    for i, (exp, det, sz) in enumerate(zip(df_plot['experiment'], df_plot['detection_increase'], df_plot['sahi_slice_size'])):
        axes[1].annotate(f'slice={sz}', (i, det), textcoords="offset points", 
                         xytext=(0, 5), ha='center', fontsize=8)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = output_dir / 'sahi_comparison.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"📊 Figure saved to: {fig_path}")
    
    plt.show()

In [ ]:
# Compare by experiment type (trends)
if len(df_plot) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Image size effect
    imgsz_exps = df_plot[df_plot['experiment'].str.contains('imgsz|baseline')].copy()
    if len(imgsz_exps) > 0:
        imgsz_exps = imgsz_exps.sort_values('imgsz')
        axes[0].plot(imgsz_exps['imgsz'], imgsz_exps['mAP50'], 'o-', label='mAP50', markersize=10)
        axes[0].plot(imgsz_exps['imgsz'], imgsz_exps['recall'], 's-', label='Recall', markersize=10)
        axes[0].set_xlabel('Image Size')
        axes[0].set_ylabel('Score')
        axes[0].set_title('Effect of Image Size')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    # Epochs effect
    epoch_exps = df_plot[df_plot['experiment'].str.contains('epochs|baseline')].copy()
    if len(epoch_exps) > 0:
        epoch_exps = epoch_exps.sort_values('epochs')
        axes[1].plot(epoch_exps['epochs'], epoch_exps['mAP50'], 'o-', label='mAP50', markersize=10)
        axes[1].plot(epoch_exps['epochs'], epoch_exps['recall'], 's-', label='Recall', markersize=10)
        axes[1].set_xlabel('Epochs')
        axes[1].set_ylabel('Score')
        axes[1].set_title('Effect of Training Epochs')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    # Model size effect
    model_order = ['yolo26n.pt', 'yolo26s.pt', 'yolo26m.pt', 'yolo26l.pt']
    model_exps = df_plot[df_plot['experiment'].str.contains('model|baseline')].copy()
    if len(model_exps) > 0:
        model_exps['model_order'] = model_exps['model'].apply(lambda x: model_order.index(x) if x in model_order else -1)
        model_exps = model_exps.sort_values('model_order')
        x_pos = range(len(model_exps))
        axes[2].plot(x_pos, model_exps['mAP50'], 'o-', label='mAP50', markersize=10)
        axes[2].plot(x_pos, model_exps['recall'], 's-', label='Recall', markersize=10)
        axes[2].set_xticks(x_pos)
        axes[2].set_xticklabels(['nano', 'small', 'medium', 'large'][:len(model_exps)])
        axes[2].set_xlabel('Model Size')
        axes[2].set_ylabel('Score')
        axes[2].set_title('Effect of Model Size')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save
    fig_path = output_dir / 'experiment_trends.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"📊 Figure saved to: {fig_path}")
    
    plt.show()

## 9. Run SAHI on Existing Models (Optional)

If you've already trained models and want to evaluate them with SAHI at different slice sizes.

In [ ]:
# OPTIONAL: Test different SAHI slice sizes on a single trained model
RUN_SLICE_SIZE_TEST = False

if RUN_SLICE_SIZE_TEST:
    # Path to your best trained model
    model_path = "jupiter_inlet_experiments/baseline_s_640_e100/weights/best.pt"  # Update this!
    
    slice_sizes = [256, 384, 512, 640, 768, 1024]
    slice_results = []
    
    print(f"Testing different SAHI slice sizes on: {model_path}\n")
    
    for slice_size in slice_sizes:
        print(f"Testing slice_size={slice_size}...")
        
        result = evaluate_with_sahi(
            model_path,
            val_images_dir,
            slice_size=slice_size,
            overlap_ratio=SAHI_OVERLAP_RATIO,
            conf_thresh=SAHI_CONFIDENCE_THRESHOLD
        )
        
        result['slice_size'] = slice_size
        slice_results.append(result)
        
        print(f"  Detections: {result['total_detections']}, Time: {result['time_per_image']:.3f}s/img")
    
    # Display results
    df_slices = pd.DataFrame(slice_results)
    print("\n" + "="*60)
    print("SAHI SLICE SIZE COMPARISON")
    print("="*60)
    print(df_slices[['slice_size', 'total_detections', 'time_per_image']].to_string(index=False))
    
    # Plot
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    ax1.plot(df_slices['slice_size'], df_slices['total_detections'], 'o-', color='steelblue', markersize=10)
    ax1.set_xlabel('Slice Size')
    ax1.set_ylabel('Total Detections', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')
    
    ax2 = ax1.twinx()
    ax2.plot(df_slices['slice_size'], df_slices['time_per_image'], 's-', color='coral', markersize=10)
    ax2.set_ylabel('Time per Image (s)', color='coral')
    ax2.tick_params(axis='y', labelcolor='coral')
    
    plt.title('SAHI Performance vs Slice Size')
    plt.tight_layout()
    plt.show()

## 10. Next Steps

Based on your results, consider:

1. **If larger image size helped:** Try even larger (1280, 1536)
2. **If more epochs helped:** The model may still be improving; try 300+
3. **If larger model helped:** Try yolo26x (extra large) if you have GPU memory
4. **If SAHI helped significantly:** Use SAHI for inference in production
5. **Combine best settings:** Once you know which individual changes help, combine them

In [ ]:
# OPTIONAL: Run a combined "best" experiment based on your findings
# Uncomment and modify based on your results

# RUN_COMBINED = False
# 
# if RUN_COMBINED:
#     print("Running combined best configuration...")
#     
#     model = YOLO('yolo26m.pt')  # Best model size
#     
#     results = model.train(
#         data=data_yaml_path,
#         epochs=150,      # Best epoch count
#         imgsz=1024,      # Best image size
#         batch=8,
#         rect=True,
#         device=0,
#         mosaic=0.5,
#         scale=0.2,
#         fliplr=0.5,
#         patience=30,
#         save=True,
#         plots=True,
#         project='jupiter_inlet_experiments',
#         name='combined_best',
#     )
#     
#     # Evaluate with SAHI
#     best_path = Path(results.save_dir) / 'weights' / 'best.pt'
#     sahi_result = evaluate_with_sahi(best_path, val_images_dir, slice_size=1024)
#     
#     print(f"\nCombined experiment results:")
#     print(f"  mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")
#     print(f"  Recall: {results.results_dict['metrics/recall(B)']:.4f}")
#     print(f"  SAHI detections: {sahi_result['total_detections']}")